# 01 Workspace, Notebooks and Git Folders

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Organize work in a Databricks workspace like a team does: folders and permissions, notebooks vs workspace files vs Python modules, Git folders for version control, and how code in one notebook reuses code from another (<code>%run</code>, imports from <code>.py</code> files).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Pipelines that start as one notebook quickly become dozens of notebooks and helper modules. How you structure them decides whether they're testable, reviewable and deployable. The certification's platform and CI/CD domains expect you to know Git folders, workspace files and how code is shared and promoted.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What lives where</b><br>
| Object | What it is | Typical use |<br>|---|---|---|<br>| <b>Notebook</b> | Cells of code and markdown, stored as <code>.ipynb</code> or source format | Exploration, pipeline steps, lessons like this one |<br>| <b>Workspace file</b> | Any file in the workspace: <code>.py</code>, <code>.sql</code>, <code>.yml</code>, <code>.json</code>, <code>.md</code> | Python modules, config, README files |<br>| <b>Folder</b> | Groups notebooks and files, with permissions | Team and project organization |<br>| <b>Git folder</b> | A folder linked to a remote Git repository and branch | Version control, code review, CI/CD |<br>| <b>Volume</b> (Unity Catalog) | Governed storage for data files | Raw files, exports, ML artifacts, not code |<br><br>
Since Databricks Runtime 13+, a notebook can <b>import Python modules</b> from workspace files in the same Git folder (or added to <code>sys.path</code>), just like a normal Python project.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A data team keeps one Git folder per project: <code>notebooks/</code> for pipeline entry points, <code>src/</code> for Python modules with the transformation logic, <code>tests/</code> for unit tests, and <code>databricks.yml</code> for deployment. Engineers work on feature branches in their own Git folders, open pull requests on GitHub, and CI runs the tests before merging. Notebooks stay thin and call functions from <code>src/</code>.
</div>

## Recommended project layout

```
my_project/                      <- Git folder (linked to GitHub)
├── README.md
├── databricks.yml               <- Asset Bundle: jobs and environments (07_workflows)
├── notebooks/
│   ├── 01_bronze_ingest.ipynb   <- thin entry points: parameters + calls into src/
│   ├── 02_silver_orders.ipynb
│   └── 03_gold_sales.ipynb
├── src/
│   └── my_project/
│       ├── __init__.py
│       ├── transforms.py        <- pure functions: DataFrame in, DataFrame out
│       └── quality.py
└── tests/
    └── test_transforms.py       <- pytest, runs locally and in CI
```

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
Keep <b>logic</b> in <code>.py</code> modules (testable, reviewable diffs) and <b>orchestration</b> in notebooks (parameters, calls, display). That one habit makes CI/CD and unit testing straightforward.
</div>

## 1. Where am I? Notebook context

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the notebook's path, the current user and the workspace URL from the notebook context.<br><br>
<b>Why it matters</b><br>Pipelines often need to know their own location, for example to build paths to config files next to the notebook, or to log which notebook produced a table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your notebook path (something like <code>/Workspace/Users/&lt;you&gt;/databricks-learning/04_databricks/01_workspace_notebooks_repos</code>), your user name, and the workspace host.
</div>

In [0]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
print("notebook path :", ctx.notebookPath().get())
print("user          :", spark.sql("SELECT current_user()").first()[0])
print("workspace url :", spark.conf.get("spark.databricks.workspaceUrl", "not available"))

## 2. Workspace files: write a Python module and import it

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small Python module as a workspace file next to this notebook, adds its folder to <code>sys.path</code>, and imports a function from it.<br><br>
<b>Why it matters</b><br>This is the modern way to share code: real Python modules, importable from any notebook, testable with <code>pytest</code>, and diffable in Git. It replaces most uses of <code>%run</code>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Workspace files are written here with plain Python file I/O. In a Git folder, commit the new <code>.py</code> file like any other file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A file <code>helpers_demo.py</code> created next to the notebook, then <code>Cleaned: 'AMAL RAJ'</code> from the imported function.
</div>

In [0]:
import os, sys, importlib

notebook_dir = os.path.dirname("/Workspace" + ctx.notebookPath().get()) if not ctx.notebookPath().get().startswith("/Workspace") \
    else os.path.dirname(ctx.notebookPath().get())
module_path = os.path.join(notebook_dir, "helpers_demo.py")

with open(module_path, "w") as f:
    f.write('''"""Tiny demo module created by 04_databricks/01."""

def clean_name(s):
    """Trim, collapse spaces and upper-case a name."""
    return " ".join(s.split()).upper() if s is not None else None
''')
print("wrote", module_path)

if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)
import helpers_demo
importlib.reload(helpers_demo)                # pick up changes if you edit the file and re-run
print("Cleaned:", repr(helpers_demo.clean_name("  amal   raj ")))

## 3. Use the module inside Spark

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the imported Python function inside a pandas UDF, and the same logic as built-in functions.<br><br>
<b>Why it matters</b><br>Module code runs on the driver when called directly, and on the executors when used in UDFs. On serverless, workspace files in the notebook's folder are available to the executors too. For DataFrame logic, prefer modules that return <b>Column expressions or DataFrame transformations</b> rather than UDFs (lesson 01_20).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with the raw names, the UDF-cleaned names and the built-in version, all equal.
</div>

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

@pandas_udf("string")
def clean_name_udf(s: pd.Series) -> pd.Series:
    import helpers_demo
    return s.map(helpers_demo.clean_name)

names = spark.createDataFrame([("  amal   raj ",), ("john smith",), (None,)], "name STRING")
names.select(
    "name",
    clean_name_udf("name").alias("via_module_udf"),
    F.upper(F.trim(F.regexp_replace("name", r"\s+", " "))).alias("via_builtins"),
).show(truncate=False)

## 4. A reusable transformation module (the recommended pattern)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a module whose functions take and return DataFrames, and calls it from the notebook.<br><br>
<b>Why it matters</b><br>Functions like <code>add_audit_columns(df)</code> or <code>clean_customers(df)</code> are the units you unit-test and reuse across notebooks and jobs. The notebook just wires them together.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The cleaned customers with audit columns added by the module.
</div>

In [0]:
with open(os.path.join(notebook_dir, "transforms_demo.py"), "w") as f:
    f.write('''from pyspark.sql import DataFrame, functions as F

def clean_customers(df: DataFrame) -> DataFrame:
    """Standardize names and countries."""
    return (df.withColumn("name", F.initcap(F.trim("name")))
              .withColumn("country", F.upper(F.trim("country"))))

def add_audit_columns(df: DataFrame, source: str) -> DataFrame:
    """Add the standard audit columns."""
    return df.withColumns({"source_system": F.lit(source), "processed_at": F.current_timestamp()})
''')

import transforms_demo
importlib.reload(transforms_demo)

raw = spark.createDataFrame([(" amal ", "in"), ("JOHN", " uk")], "name STRING, country STRING")
display(raw.transform(transforms_demo.clean_customers).transform(transforms_demo.add_audit_columns, "crm"))

## 5. `%run` vs imports

| | `%run ./other_notebook` | `import my_module` (workspace file) |
|---|---|---|
| What it shares | Everything defined in the other notebook, in the same context | Only what you import |
| Unit testable | Hard | Yes, with `pytest` |
| Git diffs | Notebook JSON | Clean Python |
| Parameters | None | Function arguments |
| Use for | Quick shared setup, legacy code | Production logic |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
<code>dbutils.notebook.run()</code> is a third option: it runs another notebook as a separate job step with parameters and returns a value (lesson 07_06).
</div>

## 6. Git folders: the daily workflow

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Summarizes how to work with Git folders, as practised in <code>00_04_git_workflow</code>.<br><br>
<b>Why it matters</b><br>Every change to production code should go through a branch and a pull request. Git folders make that possible without leaving Databricks.
</div>

| Step | Where |
|---|---|
| Clone a repo | Workspace → Create → Git folder |
| Create or switch branch | Git dialog (branch name button) |
| Commit and push | Git dialog → select files → message → Commit & Push |
| Pull changes | Git dialog → Pull |
| Resolve conflicts | Git dialog shows conflicts, or resolve on GitHub |
| Production | Jobs point at a Git branch or a deployed bundle, never at a personal folder |

## 7. Cleanup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Removes the demo modules created in this lesson.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>removed</code> for each file.
</div>

In [0]:
for name in ["helpers_demo.py", "transforms_demo.py"]:
    p = os.path.join(notebook_dir, name)
    if os.path.exists(p):
        os.remove(p)
        print("removed", name)

## Under the hood

```
Notebook (serverless)                     Workspace file system
  sys.path includes the notebook's  ───▶  /Workspace/Users/<you>/project/src/...
  folder (Git folder root on DBR 14+)      .py files are normal files you can open/import
  import transforms_demo  ───────────────▶ module loaded on the driver (Python client)
  pandas UDF using the module ──────────▶ module files shipped / available to executor Python workers
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> creating notebooks, files and folders doesn't use compute.

The workspace stores notebooks and files. **Data** belongs in Unity Catalog tables and volumes, not in workspace folders: workspace files are for code and small config only (size limits apply).

In [0]:
# Pure-Spark check of the same cleaning logic (runs anywhere)
from pyspark.sql import functions as F
spark.createDataFrame([("  amal   raj ",)], "name STRING") \
    .select(F.upper(F.trim(F.regexp_replace("name", r"\s+", " "))).alias("clean")).show()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>ModuleNotFoundError</code> when importing a workspace file</b><br>
The module's folder isn't on <code>sys.path</code>. Append it, or keep the module at the Git folder root (on by default for Git folders).<br><br>
<b>⛔ Problem: changes to a module don't show up</b><br>Python caches imported modules. Use <code>importlib.reload(module)</code> or detach and reattach.<br><br>
<b>⛔ Problem: large data files committed to Git</b><br>Keep data in volumes and tables. Add data paths to <code>.gitignore</code>.<br><br>
<b>⛔ Problem: a job runs code from someone's personal folder</b><br>Point jobs at a Git branch or a deployed bundle, so production code is reviewed and versioned.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you structure code in a Databricks project?</b><br>
In a Git folder: thin notebooks as entry points that read parameters and call functions, Python modules in <code>src/</code> with the transformation logic, unit tests in <code>tests/</code>, and an Asset Bundle definition for jobs and environments. Data lives in Unity Catalog tables and volumes, not in the workspace.<br><br>

<b>🎤 2. What is the difference between <code>%run</code> and importing a module?</b><br>
<code>%run</code> executes another notebook in the current context, sharing all its variables. Importing a workspace <code>.py</code> module shares only what you import, works with standard Python tooling, and is unit-testable, so it's preferred for production logic.<br><br>

<b>🎤 3. What are Git folders?</b><br>
Workspace folders linked to a remote Git repository, where you can switch branches, commit, push and pull from the Databricks UI. They're the basis for code review and CI/CD.<br><br>

<b>🎤 4. What are workspace files?</b><br>
Non-notebook files stored in the workspace, such as <code>.py</code>, <code>.sql</code>, <code>.yml</code> or <code>.json</code>. They let you build normal Python packages and keep config next to notebooks.<br><br>

<b>🎤 5. Where should data files go?</b><br>
In Unity Catalog volumes (files) and tables (tabular data), with governance and lineage, not in workspace folders or Git.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer wants transformation logic that can be unit tested with <code>pytest</code> and reused by several notebooks. What is the best approach?</b><br>
A. Put the logic in a notebook and use <code>%run</code> in each notebook<br>
B. Put the logic in a Python module (workspace file) in the Git folder and import it<br>
C. Copy the code into each notebook<br>
D. Store the code in a Delta table<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which Databricks feature lets a team commit, branch and pull notebooks from a remote repository inside the workspace?</b><br>
A. Unity Catalog<br>
B. Git folders<br>
C. Volumes<br>
D. Delta Sharing<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a module <code>quality_demo.py</code> next to this notebook with a function <code>null_counts(df)</code> that returns a one-row DataFrame of null counts per column</li><li>Import it and run it on a small DataFrame with some nulls</li><li>Change the function to also return the total row count, reload the module, and run it again</li><li>In two sentences: why should production jobs not run notebooks from personal folders?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1-2. Module with a DataFrame function, then import and use it
with open(os.path.join(notebook_dir, "quality_demo.py"), "w") as f:
    f.write('''from pyspark.sql import functions as F

def null_counts(df):
    return df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
''')
import quality_demo
importlib.reload(quality_demo)
sample = spark.createDataFrame([(1, None), (None, "x"), (3, "y")], "a INT, b STRING")
quality_demo.null_counts(sample).show()

# 3. Change the module and reload it
with open(os.path.join(notebook_dir, "quality_demo.py"), "w") as f:
    f.write('''from pyspark.sql import functions as F

def null_counts(df):
    return df.select(F.count("*").alias("rows"), *[F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
''')
importlib.reload(quality_demo)
quality_demo.null_counts(sample).show()
os.remove(os.path.join(notebook_dir, "quality_demo.py"))

# 4. Personal folders aren't reviewed or versioned, and anyone can change them at any time;
#    production should run reviewed code from a Git branch or a deployed bundle.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Notebooks for orchestration and exploration. Python modules (workspace files) for logic</li><li>Git folders give branches, commits and pull requests inside Databricks</li><li>Import modules from the Git folder (add the folder to <code>sys.path</code> if needed), and <code>importlib.reload</code> after edits</li><li><code>%run</code> shares a whole notebook context. Imports share only what you need</li><li>Data goes in Unity Catalog tables and volumes, not in the workspace or Git</li><li>Production jobs run reviewed code from a branch or bundle</li></ul>
</div>

| Task | Code |
|---|---|
| Notebook path | `dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()` |
| Import a module | `sys.path.append(folder); import my_module` |
| Reload after edits | `importlib.reload(my_module)` |
| Include a notebook | `%run ./other_notebook` |
| Chain transformations | `df.transform(func).transform(func2, arg)` |

## Git commit

```
git add 04_databricks/01_workspace_notebooks_repos.ipynb
git commit -m "add 04_01 workspace notebooks repos notebook"
```